### Ejercicio 1: Combinaciones Lineales y Retorno Esperado

#### Planteamiento Matemático
El rendimiento esperado de un portafolio de $n$ activos, $E[R_p]$, es una combinación lineal de los rendimientos individuales de cada activo, ponderada por su peso de asignación en la cartera:

$$E[R_p] = \sum_{i=1}^n w_i E[R_i] = w^T \cdot R$$

Donde:
* $w = [w_1, w_2, w_3]^T$ es el vector de pesos que cumple con la restricción presupuestaria de sumar 1 ($\sum_{i=1}^n w_i = 1$).
* $R = [E[R_1], E[R_2], E[R_3]]^T$ es el vector de retornos esperados de los activos.

---
- Concepto y Enfoque del Libro
 - El concepto: El cálculo se fundamenta en el producto interno estándar (o dot product) en el espacio euclidiano $\mathbb{R}^n$.
 - La trampa de NumPy: En la Sección 3.1.1, Danka nos advierte sobre un error muy común al implementar el producto punto de forma casera `(np.sum(x * y))`. Debido al fenómeno de broadcasting (donde NumPy expande automáticamente dimensiones incompatibles de forma silenciosa), un vector desalineado podría correr sin errores pero arrojar un cálculo numérico completamente incorrecto. Para evitar este comportamiento destructivo en producción, el autor recomienda utilizar estrictamente la función optimizada np.dot() o el operador @.


In [6]:
import numpy as np 
# 1. Definir los retornos esperados de 3 activos (ej: Acciones A, B, y C)
# Representan el 12%, 8% y 5% de retorno esperado respectivamente
R = np.array([0.12, 0.08, 0.05])
# 2. Definir los pesos del portafolio (deben sumar exactamente 1.0)
w = np.array([0.40, 0.30, 0.30])
# 1r condicion, la suma de los pesos debe dar exactamente 1 
# y no hay posiciones cortas 
if np.isclose(np.sum(w),1) and np.all(w>=0):
    print('portafolio valido')
    # retorno esperado del portafolio 
    E_r = R @ w
    print(f"retorno esperado: {E_r:.2%}")
# evitamos posiciones cortas 
else: 
    print('los pesos no dan 1')


portafolio valido
retorno esperado: 8.70%


### Ejercicio 2: Multiplicación de Matrices y Riesgo Marginal de los Activos

#### Planteamiento Matemático
En la optimización de portafolios, multiplicar la matriz de covarianza de retornos históricos ($\Sigma$) por el vector de pesos de asignación ($w$) mapea la estructura de variabilidad conjunta del mercado hacia la contribución de riesgo de cada activo: 

$ R_{\text{marginal}} = \Sigma \cdot w $

Donde:
* $ \Sigma \in \mathbb{R}^{3 \times 3} $ es una matriz simétrica de varianzas y covarianzas.
* $w \in \mathbb{R}^{3 \times 1}$ es el vector columna de pesos.

El vector resultante $R_{\text{marginal}} \in \mathbb{R}^{3 \times 1}$ indica cómo aporta cada activo al riesgo del portafolio global en función de su correlación con los demás componentes.

---

####  Concepto y Enfoque del Libro
* **El concepto**: En la Sección 3.2, el autor define formalmente la multiplicación matricial $A B$ como la composición de transformaciones lineales, donde cada columna de la matriz resultante es una combinación lineal de las columnas de la matriz original.
* **La dimensión en la práctica**: En la Sección 3.2.4, el texto nos enseña a resolver la diferencia entre vectores unidimensionales y vectores columna bidimensionales en NumPy. 

 **Buenas prácticas en desarrollo:**
Para evitar "abusos creativos de notación" que rompen los algoritmos en producción, debemos moldear los vectores de manera explícita como vectores columna utilizando `.reshape(-1, 1)`. Para multiplicar la matriz por el vector columna de manera óptima, el libro nos orienta a usar la función nativa `np.matmul()` o su equivalente directo en sintaxis: el operador `@`.


In [12]:
import numpy as np
# 1. Matriz de covarianza de activos (3x3)
# La diagonal representa las varianzas individuales; fuera de ella están las covarianzas
Sigma = np.array([[0.040, 0.005, 0.010],
                  [0.005, 0.025, -0.002],
                  [0.010, -0.002, 0.015]
                  ])

# 2. Vector de pesos del portafolio
w_raw = np.array([0.40, 0.30, 0.30])

# como calcularemos el riesgo marginal, usaremos una multiplicacion matricial explicita
w = w_raw.reshape(-1,1)
R_marginal = Sigma @ w
print(f"Riesgo marginal del portafolio:")
R_marginal


Riesgo marginal del portafolio:


array([[0.0205],
       [0.0089],
       [0.0079]])

### Ejercicio 3: Caso Práctico Aplicado – Varianza Total del Portafolio

#### Planteamiento Matemático
Para hallar la varianza global de un portafolio de activos ($\sigma^2_p$), se utiliza una forma cuadrática que pondera de forma simultánea los pesos en relación con la matriz de covarianzas del mercado: 

$\sigma^2_p = w^T \cdot \Sigma \cdot w$

Donde:
* $w^T \in \mathbb{R}^{1 \times 3}$ es el vector fila de pesos (la transpuesta de nuestro vector columna).
* $\Sigma \in \mathbb{R}^{3 \times 3}$ es la matriz de covarianzas.
* $w \in \mathbb{R}^{3 \times 1}$ es el vector columna de pesos. 

El resultado final de esta operación matricial consecutiva es un **valor escalar único** (un número real) que representa el riesgo consolidado del portafolio.

---

#### Concepto y Enfoque del Libro
* **El concepto**: Esta estructura matemática es un caso especial de las formas bilineales definidas en el libro como funciones de la forma $B(x, y) = x^T A y$. En las lecciones de álgebra lineal del libro, específicamente en el **Problema 9 del Capítulo 3**, se desafía al lector a implementar código en NumPy para evaluar de forma genérica formas bilineales bajo esta precisa estructura matricial.
* **La transposición**: Para ejecutar $w^T$, el libro nos enseña que la transposición simplemente consiste en "voltear" la matriz sustituyendo filas por columnas. Se accede a ella en NumPy a través de la función `np.transpose(A)` o directamente con el atributo `.T` del arreglo.


In [14]:
import numpy as np
# 1. Matriz de covarianza (3x3)
Sigma = np.array([
    [0.040, 0.005, 0.010],
    [0.005, 0.025, -0.002],
    [0.010, -0.002, 0.015]
])
# 2. Vector de pesos moldeado como columna (3x1)
w = np.array([0.40, 0.30, 0.30]).reshape(-1, 1)

# Validar la suma de ponderaciones = 1 , no posiciones cortas 
if np.isclose(np.sum(w), 1) and np.all(w>=0): 
    # para ejecutar w^T 
    # w.T vector fila (1,3)
    # calcular la varianza total del portafolio 
    var_p = (w.T @ Sigma @ w).item() # calculo matricial -> escalar 
    print(f'La varianza del portafolio es : {var_p:.2%}')
    print(f'la volatilidad del portafolio es: {np.sqrt(var_p):.2%}')


La varianza del portafolio es : 1.32%
la volatilidad del portafolio es: 11.51%


# Vectores independientes vs No dependientes

 ## Combinaciones Lineales y Retornos de Portafolio1. 
 - Formulación Matemática
 - Un portafolio de inversión distribuye su capital entre $n$ activos mediante un vector de pesos $w = (w_1, w_2, \dots, w_n) \in \mathbb{R}^n$. 
 - El espacio de portafolios válidos y completamente invertidos (sin apalancamiento neto ni caja ociosa) es un hiperplano afín en $\mathbb{R}^n$. 
 - Esto impone la restricción lineal de que la suma de sus componentes sea exactamente igual a $1$, lo que matemáticamente equivale a que el producto interno euclidiano de $w$ con el vector de unos $\mathbf{1} = (1, 1, \dots, 1) \in \mathbb{R}^n$ sea unitario:
$$\langle w, \mathbf{1} \rangle = \sum_{i=1}^n w_i = 1$$

Si un vector de pesos iniciales $w_0$ no cumple esta restricción (pero $\langle w_0, \mathbf{1} \rangle \neq 0$), se proyecta sobre el hiperplano válido mediante una normalización: 
$$w = \frac{w_0}{\langle w_0, \mathbf{1} \rangle}$$

Dado un vector de retornos esperados de los activos $R \in \mathbb{R}^n$, el retorno esperado del portafolio $\mu_p \in \mathbb{R}$ es una combinación lineal de los retornos de los activos ponderados por sus pesos, descrita por el producto interno

$$\mu_p = \langle w, R \rangle = w^T R$$

Asimismo, si disponemos de una serie temporal de retornos históricos representados por una matriz $X \in \mathbb{R}^{T \times n}$ (donde $T$ es el número de pasos de tiempo), la serie de retornos históricos del portafolio $y \in \mathbb{R}^T$ se calcula mapeando el vector $w$ a través de la transformación lineal definida por $X$
$$y = Xw$$

In [8]:
import numpy as np 

def portafolio_construction(X : np.ndarray, w_init : np.ndarray): 
    """
    simular diferentes rendimientos aplicando restricciones
    Args: 
     X (np.ndarray) : Matriz de retornos historicos : (T,n)
     w_init (np.ndarray) : Matriz-unidimensional de pesos : (T,)

    Returns: 
    w_valid (np.ndarray) : vector de pesos noramlizado : (T,) 
    E_return (float) : Esperanza de rendimientos del portafolio 
    H_return (np.ndaaray) : Rendimientos historicos : (T,)
    """ 
    # calcular la media de los activos 
    mu = np.mean(X, axis= 0)
    # Restricciones si cumplen 
    suma_pesos = w_init.sum()

    if np.isclose(suma_pesos, 0):
        raise ValueError("No se pueden normalizar pesos con suma cero.")

    if np.any(w_init < 0):
        raise ValueError("No se permiten posiciones cortas.")

    # Si ya suma 1, permanece igual; si no, se normaliza radialmente.
    w_valid = w_init / suma_pesos

    mu = X.mean(axis=0)          # forma (n,)
    E_return = mu @ w_valid      # escalar
    H_return = X @ w_valid       # forma (T,)
    return w_valid, E_return, H_return


np.random.seed(42)
X_sim = np.random.normal(0.0005, 0.015, (100, 3)) # 100 días, 3 activos
w_sucio = np.array([0.5, 0.5, 0.5]) # Suma 1.5 (no cumple restricción)

w_clean, mu_p, y_p = portafolio_construction(X_sim, w_sucio)
print("Pesos válidos:", w_clean)  # Debe ser [0.333, 0.333, 0.333]
print("Retorno esperado del portafolio:", mu_p)

    


Pesos válidos: [0.33333333 0.33333333 0.33333333]
Retorno esperado del portafolio: 0.0004167716137813928


### Independencia Lineal y Factores Redundantes (Gram-Schmidt).
 Formulación Matemática
 - En la inversión multifactorial, la exposición de $n$ activos a un conjunto de $f$ factores de riesgo se modela mediante la matriz de cargas $B \in \mathbb{R}^{n \times f}$.
 -  Si dos o más factores son linealmente dependientes, es decir, existe una combinación lineal no trivial tal que:
$$\sum_{j=1}^f \alpha_j b_j = \mathbf{0} \quad \text{con algún } \alpha_j \neq 0$$
- entonces la matriz de factores es redundante, lo que provoca multicolinealidad y sobreajuste en las estimaciones. De acuerdo con el Teorema del Rango-Nulidad: 
$$\dim(\ker B) + \text{rank}(B) = f$$
- Si el rango de la matriz $B$ es estrictamente menor que el número de factores ($\text{rank}(B) < f$), entonces el núcleo $\ker B$ contiene vectores no nulos. 
- Esto significa que existen combinaciones de factores redundantes que no aportan información nueva.
- Para extraer el núcleo de factores linealmente independientes, aplicamos el proceso de ortogonalización de Gram-Schmidt11 sobre las columnas de $B$. Tal como Danka explica en el Remark 2.2.1 de su libro, si este proceso procesa un vector que es linealmente dependiente de los vectores anteriores, se produce matemáticamente un vector nulo en la salida ($e_j = \mathbf{0}$)12. En computación, debido a los errores de redondeo de coma flotante, si la norma de la columna ortogonalizada es inferior a un umbral $\epsilon \approx 10^{-8}$, la columna se considera redundante y se descarta del modelo.

In [18]:
def extraer_factores_independientes(B: np.ndarray, tol: float = 1e-8):
    n, f = B.shape
    # factores independientes 
    columnas_ortogonales = []
    indices_retenidos = []
    indices_redundantes = []
    for j in range(f):
        v = B[:, j].astype(float).copy()
        # Restar a v sus componentes en las direcciones ya aceptadas
        for e in columnas_ortogonales:
            #Proyectar v sobre el espacio generado por las columnas ya ortogonalizadas
            #  y restarlo
            proyeccion = ((e @ v) / (e @  e)) * e
            v = v - ((proyeccion))

        # si aporta informacion nueva 
        if np.linalg.norm(v) > tol:
            columnas_ortogonales.append(v)
            indices_retenidos.append(j)
        else:
            indices_redundantes.append(j)
    return B[:, indices_retenidos], indices_retenidos

# Supongamos 5 activos y 3 factores, donde el Factor 3 es combinación lineal de los primeros dos.
f1 = np.array([1.0, 2.0, 0.5, -1.0, 1.5])
f2 = np.array([0.0, 1.0, -1.0, 2.0, 0.5])
f3 = 2.0 * f1 - 0.5 * f2  # Redundante / Colineal
B_exposiciones = np.column_stack([f1, f2, f3])

B_clean, indices = extraer_factores_independientes(B_exposiciones)
print("Índices de factores linealmente independientes retenidos:", indices) # Debe ser [1]
print("Matriz de factores limpia:\n", B_clean)

Índices de factores linealmente independientes retenidos: [0, 1]
Matriz de factores limpia:
 [[ 1.   0. ]
 [ 2.   1. ]
 [ 0.5 -1. ]
 [-1.   2. ]
 [ 1.5  0.5]]


#### Ejercicio 3: Rango, Singularidad de la Matriz de Covarianza y Regularización. 

- Formulación MatemáticaLa matriz de covarianza de activos $\Sigma \in \mathbb{R}^{n \times n}$ es inherentemente simétrica y positiva semidefinida. Según el teorema de descomposición espectral1415, posee autovalores reales $\lambda_1 \ge \lambda_2 \dots \ge \lambda_n \ge 0$. Decimos que $\Sigma$ es estrictamente positiva definida si para cualquier portafolio no nulo $w \neq \mathbf{0}$16: $$\sigma_p^2 = w^T \Sigma w > 0$$Esto garantiza que la varianza de cualquier portafolio real sea estrictamente positiva y que $\langle x, y \rangle_{\Sigma} = x^T \Sigma y$ defina un producto interno válido17.Sin embargo, en la práctica (por ejemplo, si el número de activos $n$ es mayor que el número de observaciones de retorno históricas $T$, o si existen activos con colinealidad perfecta), $\Sigma$ puede ser singular ($\det(\Sigma) = 0$), lo que significa que posee columnas linealmente dependientes18 y que su autovalor mínimo es cero ($\lambda_n = 0$).Esto daña gravemente la Optimización de Varianza Mínima Global, que busca resolver: $$\min w^T \Sigma w \quad \text{sujeto a} \quad w^T \mathbf{1} = 1$$La solución analítica requiere que $\Sigma$ sea invertible (rango completo)1920: $$w^* = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^T\Sigma^{-1}\mathbf{1}}$$Si $\Sigma$ es singular, existe un portafolio no trivial $w_0 \in \ker \Sigma$ tal que $\Sigma w_0 = \mathbf{0}$10. Esto implica que la varianza de este portafolio es $w_0^T \Sigma w_0 = 0$, lo que el optimizador interpretará erróneamente como una oportunidad de "arbitraje sin riesgo", asignando pesos extremos e inestables numéricamente.Para solucionarlo, aplicamos regularización de Tikhonov (Ridge) agregando una perturbación en la diagonal de la matriz, controlada por un parámetro $\alpha > 0$: $$\Sigma_{\text{reg}} = \Sigma + \alpha I$$Matemáticamente, esto desplaza todos los autovalores $\lambda_i$ a $\lambda_i + \alpha > 0$, haciendo que la matriz sea estrictamente positiva definida y, por ende, invertible19: $$w^T \Sigma_{\text{reg}} w = w^T \Sigma w + \alpha \|w\|_2^2 \ge \alpha \|w\|_2^2 > 0 \quad (\forall w \neq 0)$$